# Local alternative feature summary

This notebook demonstrates `plotly.local.alternative_feature_summary` for one local alternative explanation. The plot is a local summary: it shows how often each feature participates in the alternatives for one explained instance. It is not global feature importance.

The main stacked bars show primary role plus quality-flag combinations such as `counter + ensured`, `counter + pareto`, and `counter + ensured + pareto`. `ensured` and `pareto` are quality flags represented inside the role bars, not a separate default status panel.

The optional conjunction panel is disabled by default. When enabled, it counts how often a feature participates in multi-feature rules. Unknown roles mean the role metadata was unavailable or unmapped, not that the rule has no role.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# NOTE: calibrated-explanations>=1.0.0rc2 is not yet published to PyPI, so this editable install will fail until CE 1.0.0rc2 is released. Run this cell only in a dev environment where a matching CE build is already installed (e.g. an editable checkout); otherwise skip it and rely on the environment's already-installed package.
# import subprocess
# import sys
# from pathlib import Path

# package_dir = Path.cwd().resolve()
# if package_dir.name == 'examples':
#     package_dir = package_dir.parent
# else:
#     repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
#     if repo_candidate.exists():
#         package_dir = repo_candidate

# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])


In [3]:
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.plugin import register_plotly_visualization_components
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

register_plotly_visualization_components()  # explicit, idempotent registration
np.set_printoptions(precision=3, suppress=True)


In [4]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=0,
)

x_proper, x_holdout, y_proper, y_holdout = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=0,
    stratify=y,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_holdout,
    y_holdout,
    test_size=0.5,
    random_state=0,
    stratify=y_holdout,
)

assert len(x_proper) == 300
assert len(x_cal) == 100
assert len(X_query) == 100

In [5]:
model = RandomForestClassifier(n_estimators=100, random_state=0)
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

In [6]:
alternatives = explainer.explore_alternatives(X_query[:5])

Default view: role-quality combinations only.

In [7]:
alt = alternatives[0].plot(style="plotly.local.alternative_feature_summary", show=True)

Limit the display to the most involved features.

In [8]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    filter_top_features=5,
)

Normalize each feature row to shares while preserving raw counts in hover.

In [9]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    normalize="share",
)

Enable the optional conjunction panel. Conjunction bars count how often a feature appears in multi-feature rules.

In [10]:
alternatives.add_conjunctions(max_rule_size=7)

CalibratedExplanations(5 explanations):
Prediction [ Low ,  High]
0.750 [0.667, 1.000]
Value : Feature                                  Prediction [ Low  ,  High ]
0.31
-0.42
-2.1: 6 < 0.30 & 
0 > 0.40 & 
2 > -1.04         0.088     [ 0.012,  0.095]
-0.42
-2.1: 0 > 0.40 & 
2 > -1.04                     0.090     [ 0.015,  0.097]
-0.42
1.1
-2.1: 0 > 0.40 & 
5 < -0.67 & 
2 > -1.04        0.090     [ 0.022,  0.097]
-2.1  : 2 > -1.04                                 0.101     [ 0.022,  0.109]
0.26
-0.42
-2.1: 3 < -1.19 & 
0 > 0.40 & 
2 > -1.04        0.105     [ 0.019,  0.115]
0.26
0.31
-0.42
-2.1: 3 < -1.19 & 
6 < 0.30 & 
0 > 0.40 & 
2 > -1.04  0.106     [ 0.033,  0.115]
-0.07
0.31
-0.42
-2.1: 1 < -1.57 & 
6 < 0.30 & 
0 > 0.40 & 
2 > -1.04  0.115     [ 0.033,  0.126]
0.26
-0.42
1.1
-2.1: 3 < -1.19 & 
0 > 0.40 & 
5 < -0.67 & 
2 > -1.04  0.117     [ 0.046,  0.127]
1.1
-2.1: 5 < -0.67 & 
2 > -1.04                    0.125     [ 0.042,  0.137]
-0.07
-0.42
1.1
-2.1: 1 < -1.57 & 
0 > 0.40 & 
5 <

In [11]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
)

In [12]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=True,
    include_conjunctions=True,
)

Export to HTML without displaying the figure.

In [13]:
alt = alternatives[0].plot(
    style="plotly.local.alternative_feature_summary",
    show=False,
    path="alternative_feature_summary.html",
)